# Telco Customer Churn — Problem Statement and Goal

**Problem statement:** Predict which customers are likely to churn (leave the service) using the Telco dataset.

**Goal:** Build a reusable scikit-learn Pipeline that preprocesses data, tunes models (Logistic Regression and Random Forest) with GridSearchCV, evaluates performance, and exports the final pipeline for production use.

In [ ]:
# Imports: data, preprocessing, models, and utilities
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score
import joblib

In [ ]:
# Show all columns when inspecting the dataset
pd.set_option('display.max_columns', None)

In [59]:
# Load dataset
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Check the dataset shape
df.shape

(7043, 21)

In [ ]:
# Drop the customer ID column because it is only an identifier and not a predictive feature
df.drop(columns=['customerID'], inplace=True)

In [ ]:
# Check for missing values before cleaning
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [ ]:
# Inspect the number of unique values in each column
df.nunique()

gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

In [ ]:
# Converting categorical columns with 2 unique values to 0/1
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,1,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.5,0
2,1,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,1,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,0,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [ ]:
# Inspect column types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   int64  
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   int64  
 3   Dependents        7043 non-null   int64  
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   int64  
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   int64  
 16  PaymentMethod     7043 non-null   object 


In [ ]:
# Convert TotalCharges to numeric so blanks become missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# Verify the datatype change after converting TotalCharges
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   int64  
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   int64  
 3   Dependents        7043 non-null   int64  
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   int64  
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   int64  
 16  PaymentMethod     7043 non-null   object 


In [ ]:
# Remove rows with missing values created by the TotalCharges conversion
df.dropna(inplace=True)

In [ ]:
# Summarize the cleaned numeric columns
df.describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn
count,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.504693,0.162400,0.482509,0.298493,32.421786,0.903299,0.592719,64.798208,2283.300441,0.265785
std,0.500014,0.368844,0.499729,0.457629,24.545260,0.295571,0.491363,30.085974,2266.771362,0.441782
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,18.250000,18.800000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,0.000000,35.587500,401.450000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,29.000000,1.000000,1.000000,70.350000,1397.475000,0.000000
75%,1.000000,0.000000,1.000000,1.000000,55.000000,1.000000,1.000000,89.862500,3794.737500,1.000000
max,1.000000,1.000000,1.000000,1.000000,72.000000,1.000000,1.000000,118.750000,8684.800000,1.000000


In [36]:
# Define feature lists for numeric and categorical processing
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.drop('Churn').tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numeric_features, categorical_features[:10]  # show samples

(['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'tenure',
  'PhoneService',
  'PaperlessBilling',
  'MonthlyCharges',
  'TotalCharges'],
 ['MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaymentMethod'])

In [37]:
# Build preprocessing pipelines for numeric and categorical data
numeric_transformer = Pipeline([('scaler', StandardScaler())])
categorical_transformer = Pipeline([('ohe', OneHotEncoder())])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Two model pipelines: Logistic Regression and Random Forest (preprocessor + estimator)
pipe_lr = Pipeline([('preprocessor', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
pipe_rf = Pipeline([('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))])

In [ ]:
# Separate features from the target before modeling
X = df.drop(columns=['Churn'])
y = df['Churn']

In [39]:
# Split data into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [52]:
# Hyperparameter grids for GridSearchCV
param_grid_lr = {
    'clf__C': [0.01, 0.1, 1.0, 10, 30, 50],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

param_grid_rf = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 8, 10, 20],
    'clf__min_samples_split': [2, 5, 8, 10]
}

In [53]:
# Grid search for Logistic Regression (optimize ROC AUC)
gs_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1)
gs_lr.fit(X_train, y_train)
print('For LR best params:', gs_lr.best_params_, ', Score:', gs_lr.best_score_)

For LR best params: {'clf__C': 30, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'} , Score: 0.8460497866172147


In [54]:
# Grid search for Random Forest
gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1)
gs_rf.fit(X_train, y_train)
print('For RF best params:', gs_rf.best_params_, ', Score:', gs_rf.best_score_)

For RF best params: {'clf__max_depth': 8, 'clf__min_samples_split': 8, 'clf__n_estimators': 200} , Score: 0.8476337590191679


In [55]:
# Evaluate both best models on test set
for name, gs in [('LogisticRegression', gs_lr), ('RandomForest', gs_rf)]:
    y_pred = gs.predict(X_test)
    y_proba = gs.predict_proba(X_test)[:, 1]
    print(f'--- {name} ---')
    print('ROC AUC:', round(roc_auc_score(y_test, y_proba), 4))
    print(classification_report(y_test, y_pred, digits=4))


--- LogisticRegression ---
ROC AUC: 0.835
              precision    recall  f1-score   support

           0     0.8481    0.8809    0.8642      1033
           1     0.6317    0.5642    0.5960       374

    accuracy                         0.7967      1407
   macro avg     0.7399    0.7226    0.7301      1407
weighted avg     0.7906    0.7967    0.7929      1407

--- RandomForest ---
ROC AUC: 0.8351
              precision    recall  f1-score   support

           0     0.8312    0.8964    0.8626      1033
           1     0.6348    0.4973    0.5577       374

    accuracy                         0.7903      1407
   macro avg     0.7330    0.6969    0.7102      1407
weighted avg     0.7790    0.7903    0.7816      1407



In [58]:
# Choose the better model by test ROC AUC and export the full pipeline
roc_lr = roc_auc_score(y_test, gs_lr.predict_proba(X_test)[:, 1])
roc_rf = roc_auc_score(y_test, gs_rf.predict_proba(X_test)[:, 1])
best_gs = gs_lr if roc_lr >= roc_rf else gs_rf
best_name = 'LogisticRegression' if roc_lr >= roc_rf else 'RandomForest'
print('Selected model:', best_name)
# export pipeline to joblib file
joblib.dump(best_gs.best_estimator_, 'best_pipeline.joblib')
print('Exported pipeline to best_pipeline.joblib')


Selected model: RandomForest
Exported pipeline to best_pipeline.joblib


# Explanation of Results and Final Insights

- **Preprocessing and setup:** The notebook cleans the Telco data by converting `TotalCharges` to numeric, removing rows with missing values, dropping `customerID`, scaling numeric features, and one-hot encoding categorical features inside the scikit-learn pipeline.
- **Model tuning:** `GridSearchCV` was run for both Logistic Regression and Random Forest. The best cross-validation ROC AUC was slightly higher for Random Forest (`0.8476`) than Logistic Regression (`0.8460`).
- **Test performance:** On the held-out test set, both models performed similarly. Logistic Regression reached ROC AUC `0.8350`, while Random Forest reached ROC AUC `0.8351`, so Random Forest was selected as the final model.
- **Export:** The selected pipeline was saved as `best_pipeline.joblib`, which includes preprocessing and the tuned classifier together.
- **Final insight:** This notebook is production-friendly because the same preprocessing steps used during training are bundled with the model. If you later score new raw data, keep the preprocessing inside the pipeline so the feature handling stays consistent.

Overall, the workflow is complete: clean the data, build the pipeline, tune both models, compare them fairly, and export the best end-to-end pipeline for reuse.